### Trabajo Práctico Final Integrador

*Tu modelo, tu API, tu web: un modelo entrenado por vos, publicado en la
nube y consumido desde un frontend propio*

Seminario de Actualización · Tecnicatura Superior en Ciencia de Datos e IA ·
IFTS N.º 18

Docente: Cynthia M. Villagra · Ciclo lectivo 2026 · Segundo cuatrimestre
Modalidad: individual (opcional en grupo de hasta 3) · Entrega: 20/11/2026, con defensa en la clase presencial
Alumno: Angel Sebastián Medina

El presente es el notebook utilizado para el entrenamiento del modelo de Regresión Lineal a desplegar en nuestro sitio web.

Para el mismo se seleccionó un dataset alojado en Kaggle, dieseñado para estimar el consumo de energía por hogar a partir de ciertas características habitacionales y algunos factores ambientales. Presenta 7 atributos en total y se encuentra dividido en particiones de entrenamiento y prueba.


#### 1. Carga del dataset y EDA

Se inicia importando el dataset desde Kaggle:

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("adilshamim8/predict-calorie-expenditure")

print("Path to dataset files:", path)

c:\Users\asmed\Documents\Técnicatura Ciencia de Datos e IA\Seminario de Actualización\proyecto_integrador\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 16.3k/16.3k [00:00<00:00, 6.51MB/s]

Extracting files...
Path to dataset files: C:\Users\asmed\.cache\kagglehub\datasets\govindaramsriram\energy-consumption-dataset-linear-regression\versions\1


Se visualizan las particiones de entrenamiento y prueba del dataset:

In [19]:
import pandas as pd
import os
train = pd.read_csv(os.path.join(path, "train_energy_data.csv"))
test = pd.read_csv(os.path.join(path, "test_energy_data.csv"))

display(train.head())
display(test.head())

,Building Type,Square Footage,Number of Occupants,Appliances Used,Average Temperature,Day of Week,Energy Consumption
0,Residential,7063,76,10,29.84,Weekday,2713.95
1,Commercial,44372,66,45,16.72,Weekday,5744.99
2,Industrial,19255,37,17,14.30,Weekend,4101.24
3,Residential,13265,14,41,32.82,Weekday,3009.14
4,Commercial,13375,26,18,11.92,Weekday,3279.17


,Building Type,Square Footage,Number of Occupants,Appliances Used,Average Temperature,Day of Week,Energy Consumption
0,Residential,24563,15,4,28.52,Weekday,2865.57
1,Commercial,27583,56,23,23.07,Weekend,4283.80
2,Commercial,45313,4,44,33.56,Weekday,5067.83
3,Residential,41625,84,17,27.39,Weekend,4624.30
4,Residential,36720,58,47,17.08,Weekday,4820.59


Se realiza un análisis exploratorio de datos para verificar que los datos están listos para entrenar el modelo. Primero la partición 'train':

In [11]:
# Primera vista del dataframe
print("Estructura de la partición train:")
print(f"Filas: {train.shape[0]}")
print(f"Columnas: {train.shape[1]}\n")

# Tipos de dato de los atributos
print("Tipos de dato en columnas:\n")
print(train.dtypes)

# Duplicados y valores NaN´s
print("\nDuplicados en dataset:")
print(f"{train.duplicated().sum()}")
print("\nPorcentaje de duplicados respecto al total (%):")
print(f"{round(((train.duplicated().sum()/len(train))*100), 2)}")
print("\nValores NaN en dataset:")
print(f"\n{train.isna().sum()}")
print("\nPorcentaje de datos NaN respecto al total (%):")
print(f"\n{round(((train.isna().sum()/len(train))*100), 2)}")

Estructura de la partición train:
Filas: 1000
Columnas: 7

Tipos de dato en columnas:

Building Type              str
Square Footage           int64
Number of Occupants      int64
Appliances Used          int64
Average Temperature    float64
Day of Week                str
Energy Consumption     float64
dtype: object

Duplicados en dataset:
0

Porcentaje de duplicados respecto al total (%):
0.0

Valores NaN en dataset:

Building Type          0
Square Footage         0
Number of Occupants    0
Appliances Used        0
Average Temperature    0
Day of Week            0
Energy Consumption     0
dtype: int64

Porcentaje de datos NaN respecto al total (%):

Building Type          0.0
Square Footage         0.0
Number of Occupants    0.0
Appliances Used        0.0
Average Temperature    0.0
Day of Week            0.0
Energy Consumption     0.0
dtype: float64


Luego la partición 'test':

In [12]:
# Primera vista del dataframe
print("Estructura de la partición 'test':")
print(f"Filas: {test.shape[0]}")
print(f"Columnas: {test.shape[1]}\n")

# Tipos de dato de los atributos
print("Tipos de dato en columnas:\n")
print(test.dtypes)

# Duplicados y valores NaN´s
print("\nDuplicados en dataset:")
print(f"{test.duplicated().sum()}")
print("\nPorcentaje de duplicados respecto al total (%):")
print(f"{round(((test.duplicated().sum()/len(test))*100), 2)}")
print("\nValores NaN en dataset:")
print(f"\n{test.isna().sum()}")
print("\nPorcentaje de datos NaN respecto al total (%):")
print(f"\n{round(((test.isna().sum()/len(test))*100), 2)}")

Estructura de la partición 'test':
Filas: 100
Columnas: 7

Tipos de dato en columnas:

Building Type              str
Square Footage           int64
Number of Occupants      int64
Appliances Used          int64
Average Temperature    float64
Day of Week                str
Energy Consumption     float64
dtype: object

Duplicados en dataset:
0

Porcentaje de duplicados respecto al total (%):
0.0

Valores NaN en dataset:

Building Type          0
Square Footage         0
Number of Occupants    0
Appliances Used        0
Average Temperature    0
Day of Week            0
Energy Consumption     0
dtype: int64

Porcentaje de datos NaN respecto al total (%):

Building Type          0.0
Square Footage         0.0
Number of Occupants    0.0
Appliances Used        0.0
Average Temperature    0.0
Day of Week            0.0
Energy Consumption     0.0
dtype: float64


Ambas particiones no presentan registros duplicados, valores faltantes ni error en los tipos de datos.

Se traducen los nombres de las columnas al español:

In [36]:
columnas_espanhol = {'Building Type' : 'Tipo de Edificio',
                    'Square Footage' : 'Metros Cuadrados',
                    'Number of Occupants' : 'Cantidad de Ocupantes',
                    'Appliances Used' : 'Cantidad de Electrodomésticos',
                    'Average Temperature' : 'Temperatura Promedio',
                    'Day of Week' : 'Día de Semana',
                    'Energy Consumption' : 'Consumo de Energía'}
test.rename(columns=columnas_espanhol, inplace=True)

In [37]:
# Copiamos el dataset para realizar cambios y dejar el original intacto
train_copy = train.copy()
train_copy.rename(columns=columnas_espanhol, inplace=True)
train_copy.head()

,Tipo de Edificio,Metros Cuadrados,Cantidad de Ocupantes,Cantidad de Electrodomésticos,Temperatura Promedio,Día de Semana,Consumo de Energía
0,Residential,7063,76,10,29.84,Weekday,2713.95
1,Commercial,44372,66,45,16.72,Weekday,5744.99
2,Industrial,19255,37,17,14.30,Weekend,4101.24
3,Residential,13265,14,41,32.82,Weekday,3009.14
4,Commercial,13375,26,18,11.92,Weekday,3279.17


In [38]:
# Se repiten los pasos para el dataset de test
test_copy = test.copy()
test_copy.rename(columns=columnas_espanhol, inplace=True)
test_copy.head()

,Tipo de Edificio,Metros Cuadrados,Cantidad de Ocupantes,Cantidad de Electrodomésticos,Temperatura Promedio,Día de Semana,Consumo de Energía
0,Residential,24563,15,4,28.52,Weekday,2865.57
1,Commercial,27583,56,23,23.07,Weekend,4283.80
2,Commercial,45313,4,44,33.56,Weekday,5067.83
3,Residential,41625,84,17,27.39,Weekend,4624.30
4,Residential,36720,58,47,17.08,Weekday,4820.59


Se realiza *Label Encoding* para la columna *Tipo de Edificio* y *One Hot Encoding* para la columna *Día de Semana* para convetirlos en datos del tipo numérico para la construcción del modelo de Regresión.

Primero la partición *train*:

In [40]:
# Label encoding de la columna 'Tipo de Edificio'
categoria_edificios = { 'Residential' : 1,
                        'Commercial' : 2,
                        'Industrial' : 3}

train_copy['Tipo de Edificio'] = train_copy['Tipo de Edificio'].map(categoria_edificios)

# One-hot encoding de la columna 'Día de Semana'
train_copy['Día de Semana'] = train_copy['Día de Semana'].map(lambda x: 1 if x == 'Weekend' else 0)

train_copy.head()

,Tipo de Edificio,Metros Cuadrados,Cantidad de Ocupantes,Cantidad de Electrodomésticos,Temperatura Promedio,Día de Semana,Consumo de Energía
0,1,7063,76,10,29.84,0,2713.95
1,2,44372,66,45,16.72,0,5744.99
2,3,19255,37,17,14.30,1,4101.24
3,1,13265,14,41,32.82,0,3009.14
4,2,13375,26,18,11.92,0,3279.17


Luego partición *test*:

In [41]:
test_copy['Tipo de Edificio'] = test_copy['Tipo de Edificio'].map(categoria_edificios)
test_copy['Día de Semana'] = test_copy['Día de Semana'].map(lambda x: 1 if x == 'Weekend' else 0)
test_copy.head()

,Tipo de Edificio,Metros Cuadrados,Cantidad de Ocupantes,Cantidad de Electrodomésticos,Temperatura Promedio,Día de Semana,Consumo de Energía
0,1,24563,15,4,28.52,0,2865.57
1,2,27583,56,23,23.07,1,4283.80
2,2,45313,4,44,33.56,0,5067.83
3,1,41625,84,17,27.39,1,4624.30
4,1,36720,58,47,17.08,0,4820.59


#### 2. Entrenamiento del Modelo

El dataset ya se haya dividido en particiones de entrenamiento (1000 registros) y prueba (100 registros), por lo que separaremos unicamente la variable objetivo ("Consumo de Energía") de las predictoras:

In [54]:
X_train, y_train = train_copy.drop('Consumo de Energía', axis=1), train_copy['Consumo de Energía']
X_test, y_test = test_copy.drop('Consumo de Energía', axis=1), test_copy['Consumo de Energía']

Antes de entrenar el modelo, debemos estandarizar las particiones. Importamos *StandardScaler()* de *scikit-learn*:

In [55]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

pipeline = make_pipeline(StandardScaler(), LinearRegression())
pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('linearregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](6,)","['Tipo de Edificio','Metros Cuadrados','Cantidad de Ocupantes', 'Cantidad de Electrodomésticos','Temperatura Promedio','Día de Semana']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,6
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


In [56]:
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

predicciones = pipeline.predict(X_test)
r2 = r2_score(y_test, predicciones)
rmse = mean_squared_error(y_test, predicciones) ** 0.5

print(f"RMSE = {rmse:5.2f} miles de US$   R² = {r2_score(y_test, predicciones):.3f}")

RMSE =  0.01 miles de US$   R² = 1.000


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

escalador = StandardScaler()
escalador.fit(X_train)

StandardScaler()

Obtenidos los parámetros con el escalador, se estandariza con *transform()*:

In [49]:
X_train_esc = escalador.transform(X_train)
X_test_esc = escalador.transform(X_test)

In [50]:
print("Primer registro original:", X_train.iloc[0].values)
print("Primer registro escalado:", X_train_esc[0])

Primer registro original: [1.000e+00 7.063e+03 7.600e+01 1.000e+01 2.984e+01 0.000e+00]
Primer registro escalado: [-1.19119324 -1.28780475  0.95113382 -1.10695673  1.0129251  -0.98609664]


Importamos la clase *LinearRegression* de *scikit-learn* y creamos una instancia para entrenar:

In [52]:
from sklearn.linear_model import LinearRegression

regresion_lineal = LinearRegression()
regresion_lineal.fit(X_train_esc, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](6,)","[407.16,714.37,290.47,281.96,-35.68,-25. ]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,4166
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,6
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(6)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](6,)","[32.84,32.63,31.66,31.2 ,30.98,30.36]"


Entrenado el modelo, vamos a evaluarlo con la partición de entrenamiento:

In [ ]:
from sklearn.metrics import r2_score

prediccio